# Thesis Pipeline Orchestrator
This notebook starts the Neo4j environment, optionally rebuilds the dataset, selects an optimized historical/detection window,
and runs the full label-aware and label-agnostic FastRP analyses. Artifacts are archived under `thesis_results/` for thesis use.

In [1]:
# Global configuration for the thesis analysis pipeline
from pathlib import Path
from datetime import datetime
import glob
import json
import shutil
import pandas as pd
import numpy as np

from CART import Controller

HISTORICAL_WINDOW_HOURS = 48
DETECTION_WINDOW_HOURS = 24
EMBEDDING_DIM = 128

REBUILD_DATABASE = False  # Download and rebuild Neo4j from source data
RUN_WINDOW_SWEEP = False   # Execute optimize_windows.py across presets
USE_OPTIMIZED_WINDOW = True  # Override above constants when sweep results exist
RUN_LABEL_AWARE = True      # Run MITRE ATT&CK-aware experiment
RUN_LABEL_AGNOSTIC = True   # Run structural (label-agnostic) experiment
LABEL_AGNOSTIC_LIMIT = None  # Optional cap for label-agnostic recon events
SHUTDOWN_AFTER_RUN = False   # Stop container when notebook completes

OUTPUT_DIR = Path('thesis_results')
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
# Start or connect to the shared Neo4j controller container
controller = Controller()

Controller initialized.
Analysis Controller (inherits Neo4jConnection) is configured and ready.


In [3]:

status = controller.status()
if not status.get('running'):
    print('Starting Neo4j container...')
    controller.start()
else:
    print('Neo4j container already running.')

if not controller.connect():
    raise RuntimeError('Could not connect to Neo4j; check container logs.')
print('Controller connected to Neo4j.')

Container 'neo4j_thesis_server': missing, unknown; driver_connected=False
Starting Neo4j container...
Browser port 7474 is IN USE.
Docker requires sudo privileges.
Container 'neo4j_thesis_server' is already running and exposes HTTP on port 7474. No new container started.
✓ Successfully connected to the Neo4j database.
Controller connected to Neo4j.


In [4]:
# Optional database rebuild (downloads the dataset and reimports into Neo4j)
if REBUILD_DATABASE:
    print('Rebuilding database with full dataset...')
    controller.build_database(rebuild=True)
else:
    print('Skipping database rebuild; set REBUILD_DATABASE=True to rebuild.')

Skipping database rebuild; set REBUILD_DATABASE=True to rebuild.


In [5]:
# Optional window sweep to regenerate comparison CSVs
if RUN_WINDOW_SWEEP:
    import optimize_windows
    optimize_windows.input = lambda prompt='': None
    optimize_windows.main()
else:
    print('Skipping window sweep; set RUN_WINDOW_SWEEP=True to execute optimize_windows.py.')

Skipping window sweep; set RUN_WINDOW_SWEEP=True to execute optimize_windows.py.


In [6]:
# Evaluate available window optimization outputs and optionally update window selection
records = []
for path in sorted(glob.glob('window_opt_*_method_comparison.csv')):
    parts = Path(path).stem.split('_')
    hist = int(parts[2])
    det = int(parts[3])
    df = pd.read_csv(path)
    if 'FastRP Embedding' not in df['Method'].values:
        continue
    row = df[df['Method'] == 'FastRP Embedding'].iloc[0]
    pred_path = path.replace('_method_comparison.csv', '_pivot_predictions.csv')
    try:
        pred_df = pd.read_csv(pred_path, usecols=['became_pivot'])
        pivot_rate = pred_df['became_pivot'].mean() * 100
        sample_count = len(pred_df)
        pivot_count = pred_df['became_pivot'].sum()
    except Exception:
        pivot_rate = np.nan
        sample_count = np.nan
        pivot_count = np.nan
    records.append({
        'historical_hours': hist,
        'detection_hours': det,
        'auc_roc': row['AUC-ROC'],
        'auc_pr': row['AUC-PR'],
        'f1': row['F1-Score'],
        'pivot_rate': pivot_rate,
        'samples': sample_count,
        'pivot_count': pivot_count
    })

if records:
    window_df = pd.DataFrame(records).sort_values(
        ['auc_roc', 'historical_hours', 'detection_hours'],
        ascending=[False, True, True]
    )
    print(window_df.to_string(index=False))
    if USE_OPTIMIZED_WINDOW:
        best_row = window_df.iloc[0]
        HISTORICAL_WINDOW_HOURS = int(best_row['historical_hours'])
        DETECTION_WINDOW_HOURS = int(best_row['detection_hours'])
        print(f"Using best window: hist={HISTORICAL_WINDOW_HOURS}h, det={DETECTION_WINDOW_HOURS}h")
else:
    print('No window optimization files found; retaining configured windows.')

print(f'Analysis window configuration: hist={HISTORICAL_WINDOW_HOURS}h, det={DETECTION_WINDOW_HOURS}h')

 historical_hours  detection_hours  auc_roc   auc_pr       f1  pivot_rate  samples  pivot_count
               48               24 0.869766 0.990443 0.973563   94.848738    28692        27214
               12               48 0.720415 0.981617 0.973563   94.848738    28692        27214
               24               12 0.715505 0.981390 0.973563   94.848738    28692        27214
               24               48 0.710609 0.981100 0.973563   94.848738    28692        27214
               12               12 0.703841 0.980693 0.973563   94.848738    28692        27214
               48               12 0.652763 0.976906 0.973563   94.848738    28692        27214
               48               48 0.644939 0.976127 0.973563   94.848738    28692        27214
               24               24 0.633053 0.975107 0.973563   94.848738    28692        27214
               12               24 0.591928 0.971782 0.973563   94.848738    28692        27214
Using best window: hist=48h, det=24h
Ana

In [7]:
# Prepare SubnetPivotAnalyzer with full-corpus reconnaissance sampling
analyzer = controller.SubnetPivotAnalyzer
if not analyzer.connect():
    raise RuntimeError('Analyzer could not connect to Neo4j.')

def patch_full_recon_sampling(analyzer_obj, label_agnostic_limit=None):
    original_fn = analyzer_obj.identify_reconnaissance_victims_by_subnet

    def patched(self, use_labels: bool, historical_window_hours: int):
        print('\n--- Identifying Reconnaissance Victims by Subnet (full corpus) ---')
        with self.driver.session(database=self.database) as session:
            if use_labels:
                query = (
                    "MATCH (a:IP)-[r:CONNECTS]->(v:IP)\n"
                    "WHERE r.is_attack = 1 AND r.tactic = 'Reconnaissance'\n"
                    "WITH DISTINCT v.subnet as victim_subnet, r.timestamp as recon_time\n"
                    "ORDER BY recon_time\n"
                    "RETURN victim_subnet, recon_time"
                )
            else:
                query = (
                    "MATCH (a:IP)-[r1:CONNECTS]->(v:IP)\n"
                    "WHERE exists { (v)-[:CONNECTS]->() }\n"
                    "WITH DISTINCT v.subnet as victim_subnet, r1.timestamp as recon_time\n"
                    "ORDER BY recon_time\n"
                    "RETURN victim_subnet, recon_time"
                )
                if label_agnostic_limit is not None:
                    query += f'\nLIMIT {int(label_agnostic_limit)}'
            result = session.run(query).data()
        print(f"  ✓ Found {len(result):,} reconnaissance events")
        return result

    analyzer_obj.identify_reconnaissance_victims_by_subnet = patched.__get__(analyzer_obj, analyzer_obj.__class__)
    return original_fn

original_recon_fn = patch_full_recon_sampling(analyzer, label_agnostic_limit=LABEL_AGNOSTIC_LIMIT)

In [8]:
# Run selected experiments with the configured window
run_modes = []
if RUN_LABEL_AWARE:
    run_modes.append('label_aware')
if RUN_LABEL_AGNOSTIC:
    run_modes.append('label_agnostic')

if not run_modes:
    raise ValueError('No experiments selected; enable RUN_LABEL_AWARE and/or RUN_LABEL_AGNOSTIC.')

def mode_artifacts_available(prefix: str) -> bool:
    pivot_path = Path(f"{prefix}_pivot_predictions.csv")
    method_path = Path(f"{prefix}_method_comparison.csv")
    missing = [p.name for p in (pivot_path, method_path) if not p.exists()]
    if missing:
        print(f"  ⚠ Missing artifacts for {prefix}: {', '.join(missing)}")
        return False
    return True

executed_prefixes = []
overall_start = datetime.utcnow()
try:
    for mode in run_modes:
        print('\n' + '=' * 80)
        print(f"Executing {mode.replace('_', ' ').title()} experiment")
        print('=' * 80)
        mode_start = datetime.utcnow()
        analyzer.run_full_analysis(
            mode=mode,
            historical_window_hours=HISTORICAL_WINDOW_HOURS,
            detection_window_hours=DETECTION_WINDOW_HOURS,
            embedding_dim=EMBEDDING_DIM
        )
        mode_finish = datetime.utcnow()
        if mode_artifacts_available(mode):
            executed_prefixes.append(mode)
        else:
            print(f"  ⚠ Skipping downstream steps for {mode}; required artifacts not produced.")
        print(f"{mode} runtime: {(mode_finish - mode_start).total_seconds():.1f} seconds")

    if {'label_aware', 'label_agnostic'}.issubset(set(executed_prefixes)):
        analyzer.compare_analysis_modes()
    else:
        print("  ⚠ Comparison skipped; ensure both modes complete successfully before comparing.")
finally:
    analyzer.identify_reconnaissance_victims_by_subnet = original_recon_fn
    overall_finish = datetime.utcnow()
    print(f"Total analysis runtime: {(overall_finish - overall_start).total_seconds():.1f} seconds")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name)'
Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPR


Executing Label Aware experiment

SUBNET-AWARE PIVOT PREDICTION WITH FASTRP EMBEDDINGS
  Mode: LABEL_AWARE
  Historical window: 48 hours
  Detection window: 24 hours
  Embedding dimension: 128

--- Adding Subnet Labels to IP Nodes ---
  Ensuring subnet strings exist...
  ✓ Processed 357 IP nodes for subnet strings
  Assigning numeric subnet IDs...
  ✓ Assigned numeric IDs to 21 distinct subnets
  ✓ Labeled 0 IPs with UNKNOWN subnet
  ✓ Subnet indices verified

RUNNING LABEL-AWARE ANALYSIS

--- Creating Graph Projection: pivot_graph_labeled ---

--- Adding Subnet Labels to IP Nodes ---
  Ensuring subnet strings exist...
  ✓ Processed 357 IP nodes for subnet strings
  Assigning numeric subnet IDs...
  ✓ Assigned numeric IDs to 21 distinct subnets
  ✓ Labeled 0 IPs with UNKNOWN subnet
  ✓ Subnet indices verified

--- Dropping All Existing Graph Projections ---
  ✓ Dropped structure projection
  ✓ Dropped labels projection
  ✓ Created structure projection: pivot_graph_labeled_structure
  

  Processing Training events: 100%|██████████| 28692/28692 [00:07<00:00, 3974.59subnet/s]



  Training Results:
    Valid samples: 28,692
    Pivots: 28,004 (97.6%)

--- Processing Test Set ---
  Pre-fetching all subnet features and pivot data...
    ✓ Loaded features for 21 subnets
    Fetching LATERAL MOVEMENT attacks for 14 unique subnets...
    Time range: 1710685406.49361 to 1711669990.118523 (273.5 hours)
    ✓ Fetched 364658 lateral movement attacks
    Processing pivot behaviors in memory...
    ✓ Loaded pivot behaviors for 28692 events
    ✓ 27214 (94.8%) are TRUE PIVOTS (lateral movement detected)
  Processing 28692 events in memory...


  Processing Testing events: 100%|██████████| 28692/28692 [00:07<00:00, 4057.41subnet/s]


...

Shared connection managed by controller; not closing driver here.
label_aware runtime: 214.3 seconds

Executing Label Agnostic experiment

SUBNET-AWARE PIVOT PREDICTION WITH FASTRP EMBEDDINGS
  Mode: LABEL_AGNOSTIC
  Historical window: 48 hours
  Detection window: 24 hours
  Embedding dimension: 128

--- Adding Subnet Labels to IP Nodes ---
  Ensuring subnet strings exist...
  ✓ Processed 357 IP nodes for subnet strings
  Assigning numeric subnet IDs...
  ✓ Assigned numeric IDs to 21 distinct subnets
  ✓ Labeled 0 IPs with UNKNOWN subnet
  ✓ Subnet indices verified

RUNNING LABEL-AGNOSTIC ANALYSIS

--- Creating Graph Projection: pivot_graph_unlabeled ---

--- Adding Subnet Labels to IP Nodes ---
  Ensuring subnet strings exist...
  ✓ Processed 357 IP nodes for subnet strings
  Assigning numeric subnet IDs...
  ✓ Assigned numeric IDs to 21 distinct subnets
  ✓ Labeled 0 IPs with UNKNOWN subnet
  ✓ Subnet indices verified

--- Dropping All Existing Graph Projections ---
  ✓ Dropped struc

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name)'


  ✓ Dropped labels projection
  ✓ Created projection: pivot_graph_unlabeled
    Nodes: 357
    Relationships: 3,797,226

--- Computing FastRP Embeddings (dim=128) ---
  ✓ Embeddings computed in 0.18s
    Nodes with embeddings: 357
    Property name: embedding_label_agnostic

--- Computing Centrality Metrics ---
  Computing PageRank...
    ✓ PageRank: 357 nodes
  Computing Betweenness Centrality (sampled)...
    ✓ Betweenness: 357 nodes in 3.03s
  Computing Clustering Coefficient...
    ✓ Clustering Coefficient: 357 nodes
      Average: 0.0571

--- Computing Temporal Features ---
  ✓ Temporal features computed
    Nodes updated: 94
    Avg connection velocity: 75.75 conn/hour
    Avg burst score: 33.5967

--- Identifying Reconnaissance Victims by Subnet (full corpus) ---
  ✓ Found 1,179,324 reconnaissance events

--- Train/Test Split ---
  Training: 589,662 events
  Testing: 589,662 events

--- Processing Training Set ---
  Pre-fetching all subnet features and pivot data...
    ✓ Loaded

  Processing Training events: 100%|██████████| 589662/589662 [01:17<00:00, 7601.64subnet/s]



  Training Results:
    Valid samples: 589,662
    Pivots: 0 (0.0%)
  ⚠ No pivots in training set
Shared connection managed by controller; not closing driver here.
  ⚠ Missing artifacts for label_agnostic: label_agnostic_pivot_predictions.csv, label_agnostic_method_comparison.csv
  ⚠ Skipping downstream steps for label_agnostic; required artifacts not produced.
label_agnostic runtime: 147.2 seconds
  ⚠ Comparison skipped; ensure both modes complete successfully before comparing.
Total analysis runtime: 361.5 seconds


In [9]:
# Collect and archive artifacts for this run
run_stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
run_dir = OUTPUT_DIR / f'run_{run_stamp}_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}'
run_dir.mkdir(parents=True, exist_ok=True)

def add_window_tag(name: str) -> str:
    if name.startswith('label_aware_'):
        return name.replace('label_aware_', f"label_aware_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_", 1)
    if name.startswith('label_agnostic_'):
        return name.replace('label_agnostic_', f"label_agnostic_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_", 1)
    return name

patterns = [f"{prefix}_*" for prefix in executed_prefixes]
patterns.append('mode_comparison.png')

moved = []
for pattern in patterns:
    for src_path in Path('.').glob(pattern):
        if not src_path.is_file():
            continue
        dest_name = add_window_tag(src_path.name)
        dest_path = run_dir / dest_name
        shutil.move(str(src_path), dest_path)
        moved.append(dest_path)
        print(f'Moved {src_path.name} -> {dest_path}')

metadata = {
    'timestamp_utc': run_stamp,
    'historical_window_hours': HISTORICAL_WINDOW_HOURS,
    'detection_window_hours': DETECTION_WINDOW_HOURS,
    'embedding_dim': EMBEDDING_DIM,
    'executed_prefixes': executed_prefixes,
    'artifacts': [str(path.name) for path in moved]
}
(run_dir / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print(f'Archived run artifacts in {run_dir}')

Moved label_aware_pivot_predictions.csv -> thesis_results/run_20251105_204225_h48_d24/label_aware_h48_d24_pivot_predictions.csv
Moved label_aware_multi_hop_chains.csv -> thesis_results/run_20251105_204225_h48_d24/label_aware_h48_d24_multi_hop_chains.csv
Moved label_aware_visualizations.png -> thesis_results/run_20251105_204225_h48_d24/label_aware_h48_d24_visualizations.png
Moved label_aware_method_comparison.csv -> thesis_results/run_20251105_204225_h48_d24/label_aware_h48_d24_method_comparison.csv
Archived run artifacts in thesis_results/run_20251105_204225_h48_d24


In [10]:
# Summarize key metrics from the archived results
def load_artifact(run_directory: Path, prefix: str, suffix: str) -> Path | None:
    pattern = f"{prefix}_h{HISTORICAL_WINDOW_HOURS}_d{DETECTION_WINDOW_HOURS}_{suffix}"
    matches = list(run_directory.glob(pattern))
    return matches[0] if matches else None

aware_method_path = load_artifact(run_dir, 'label_aware', 'method_comparison.csv') if 'label_aware' in executed_prefixes else None
agnostic_method_path = load_artifact(run_dir, 'label_agnostic', 'method_comparison.csv') if 'label_agnostic' in executed_prefixes else None
aware_preds_path = load_artifact(run_dir, 'label_aware', 'pivot_predictions.csv') if 'label_aware' in executed_prefixes else None
agnostic_preds_path = load_artifact(run_dir, 'label_agnostic', 'pivot_predictions.csv') if 'label_agnostic' in executed_prefixes else None

summary_rows = []
for label, method_path, preds_path in [
    ('Label-Aware', aware_method_path, aware_preds_path),
    ('Label-Agnostic', agnostic_method_path, agnostic_preds_path)
]:
    if method_path is None or preds_path is None:
        print(f'Missing artifacts for {label}; skip summary.')
        continue
    method_df = pd.read_csv(method_path)
    preds_df = pd.read_csv(preds_path)
    if 'FastRP Embedding' not in method_df['Method'].values:
        print(f'FastRP results missing for {label}; skip summary.')
        continue
    fastrp_row = method_df[method_df['Method'] == 'FastRP Embedding'].iloc[0]
    pivot_rate = preds_df['became_pivot'].mean() * 100 if 'became_pivot' in preds_df.columns else np.nan
    summary_rows.append({
        'Mode': label,
        'Samples': len(preds_df),
        'Pivots': preds_df['became_pivot'].sum() if 'became_pivot' in preds_df.columns else np.nan,
        'Pivot Rate (%)': pivot_rate,
        'AUC-ROC': fastrp_row['AUC-ROC'],
        'AUC-PR': fastrp_row['AUC-PR'],
        'F1-Score': fastrp_row['F1-Score'],
        'Precision': fastrp_row['Precision'],
        'Recall': fastrp_row['Recall']
    })

def to_builtin(value):
    if isinstance(value, (np.generic,)):
        return value.item()
    return value

if summary_rows:
    summary_rows_builtin = [
        {key: to_builtin(value) for key, value in row.items()}
        for row in summary_rows
    ]
    summary_df = pd.DataFrame(summary_rows_builtin)
    print(summary_df.to_string(index=False))
    summary_payload = metadata.copy()
    summary_payload['metrics'] = summary_rows_builtin
    (run_dir / 'run_summary.json').write_text(json.dumps(summary_payload, indent=2))
else:
    print('No summary generated; verify artifacts above.')

Missing artifacts for Label-Agnostic; skip summary.
       Mode  Samples  Pivots  Pivot Rate (%)  AUC-ROC   AUC-PR  F1-Score  Precision  Recall
Label-Aware    28692   27214       94.848738  0.76056 0.984486  0.973563   0.948487     1.0


In [11]:
# Optional: stop the container when finished
controller.close()
if SHUTDOWN_AFTER_RUN:
    controller.stop()
    print('Neo4j container stopped.')
else:
    print('Neo4j container left running; call controller.stop() if you want to shut it down.')

✓ Neo4j connection closed.
Neo4j container left running; call controller.stop() if you want to shut it down.
